# Notebook 01 — Data Generation & Database Load
**Project:** Multi-Donor Grant Portfolio Risk & Utilisation Analytics  
**Author:** Caroline Nzisa Mwanzia  
**Purpose:** Build simulated expenditure, disbursement, and flag tables on top of real World Bank project metadata, then load all four tables into MySQL.

---
## What This Notebook Does
1. Loads and cleans the real World Bank projects CSV → `projects` table
2. Simulates realistic monthly expenditure transactions → `expenditures` table
3. Simulates donor disbursement schedules vs actuals → `disbursements` table
4. Loads all three tables into MySQL
5. Validates row counts and runs a quick sanity check

> **Note:** The `flags` table is populated later in Notebook 03 by the risk flagging engine.

---

## Step 0 — Install & Import Libraries

In [2]:
# Run this cell once to install any missing libraries
# If already installed, it will skip cleanly
import subprocess, sys
libs = ['pandas', 'sqlalchemy', 'mysql-connector-python', 'faker', 'numpy']
for lib in libs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])
print('All libraries ready.')

All libraries ready.


In [3]:
import pandas as pd
import numpy as np
from faker import Faker
from sqlalchemy import create_engine, text
from datetime import date, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Set random seeds so results are reproducible every time you run this
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [16]:
import os

# Set this to wherever your project folder actually lives on your computer
# Based on your setup it's likely one of these:
project_root = r'C:\Users\Caroline Mwanzia\grant-portfolio-analytics'

# If you put it somewhere else e.g. Desktop or Documents, adjust accordingly:
# project_root = r'C:\Users\Caroline Mwanzia\Desktop\grant-portfolio-analytics'
# project_root = r'C:\Users\Caroline Mwanzia\Documents\grant-portfolio-analytics'

os.chdir(project_root)
print("Working directory set to:", os.getcwd())

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'C:\\Users\\Caroline Mwanzia\\grant-portfolio-analytics'

---
## Step 1 — Load & Clean the World Bank Projects Data
This uses your real downloaded CSV as the `projects` table.  
We clean column names, parse dates, and standardise country names.

In [14]:
# ── LOAD RAW DATA ─────────────────────────────────────────────────────────────
worldbank_cleaned = pd.read_csv('../data/processed/world_bank_projects_africa_cleaned.csv')

print(f'Raw data loaded: {worldbank_cleaned.shape[0]} rows, {worldbank_cleaned.shape[1]} columns')
worldbank_cleaned.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/world_bank_projects_africa_cleaned.csv'

In [ ]:
# ── BUILD THE PROJECTS TABLE ───────────────────────────────────────────────────

df_projects = pd.DataFrame()

# Keep only the columns we need and rename them to match our schema
df_projects['project_id']        = df_raw['project_id'].str.strip()
df_projects['project_name']      = df_raw['project_name'].str.strip()
df_projects['donor']             = df_raw['financing_type'].str.strip()
df_projects['country']           = df_raw['country'].str.strip()
df_projects['sector']            = df_raw['sector_1'].str.strip()
df_projects['total_budget_usd']  = df_raw['current_project_cost'].astype(float)
df_projects['status']            = df_raw['project_status'].str.strip()
df_projects['implementing_agency'] = df_raw['implementing_agency'].str.strip()

# Parse dates — World Bank format is 2017-06-20T00:00:00Z
df_projects['start_date'] = pd.to_datetime(
    df_raw['board_approval_date'], errors='coerce'
).dt.date

df_projects['end_date'] = pd.to_datetime(
    df_raw['project_closing_date'], errors='coerce'
).dt.date

# Standardise country names (remove 'Republic of' for cleaner display)
country_map = {
    'Republic of Kenya':             'Kenya',
    'Republic of Uganda':            'Uganda',
    'Republic of Rwanda':            'Rwanda',
    'United Republic of Tanzania':   'Tanzania'
}
df_projects['country'] = df_projects['country'].replace(country_map)

# Map financing type to cleaner donor labels
donor_map = {
    'IDA':    'World Bank – IDA',
    'Grants': 'World Bank – Grants',
    'Other':  'World Bank – Other Financing'
}
df_projects['donor'] = df_projects['donor'].replace(donor_map)

# Add beneficiary estimate — used later for Value-for-Money metric
# Estimated as a realistic range based on budget size and sector
def estimate_beneficiaries(row):
    budget = row['total_budget_usd']
    sector = str(row['sector']).lower()
    # Infrastructure projects reach fewer direct beneficiaries
    if any(s in sector for s in ['road', 'energy', 'water', 'transport']):
        return int(np.random.uniform(50000, 500000))
    # Social sector projects reach more people
    elif any(s in sector for s in ['health', 'education', 'social', 'protection']):
        return int(np.random.uniform(100000, 1000000))
    else:
        return int(np.random.uniform(20000, 300000))

df_projects['estimated_beneficiaries'] = df_projects.apply(estimate_beneficiaries, axis=1)

# Drop rows where dates are missing
before = len(df_projects)
df_projects = df_projects.dropna(subset=['start_date', 'end_date'])
print(f'Dropped {before - len(df_projects)} rows with missing dates.')
print(f'Projects table: {len(df_projects)} rows')

df_projects.head()

In [ ]:
# Quick validation
print('== PROJECTS TABLE SUMMARY ==')
print(f"Total projects:  {len(df_projects)}")
print(f"Countries:       {df_projects['country'].unique().tolist()}")
print(f"Donors:          {df_projects['donor'].unique().tolist()}")
print(f"Budget range:    USD {df_projects['total_budget_usd'].min():,.0f} to USD {df_projects['total_budget_usd'].max():,.0f}")
print(f"Status mix:      {df_projects['status'].value_counts().to_dict()}")
print(f"Top sectors:")
print(df_projects['sector'].value_counts().head(8).to_string())

---
## Step 2 — Simulate the Expenditures Table

**Logic:** For each project, generate monthly expenditure records across 4 categories (Staff, Operations, Procurement, Indirect) for every month between start and end date.  

**Realism rules built in:**
- Spend starts slow (ramp-up period), peaks mid-project, tapers at end
- Some projects are intentionally under-spent (to generate risk flags later)
- Some projects are intentionally over-budget in specific categories
- Staff costs are always the largest category (~40% of spend)

In [ ]:
# ── EXPENDITURE SIMULATION ─────────────────────────────────────────────────────

# Spending category proportions (must sum to 1.0)
CATEGORY_SPLITS = {
    'Staff':        0.40,
    'Operations':   0.25,
    'Procurement':  0.25,
    'Indirect':     0.10
}

# Project risk profiles — controls how much of budget gets spent
# We assign each project a profile so we get a realistic mix
RISK_PROFILES = {
    'on_track':    {'utilisation': (0.85, 0.98), 'weight': 0.55},  # 55% of projects
    'underspend':  {'utilisation': (0.40, 0.70), 'weight': 0.25},  # 25% — slow disbursement risk
    'overspend':   {'utilisation': (1.00, 1.15), 'weight': 0.20},  # 20% — budget overrun risk
}

def assign_risk_profile():
    """Randomly assign a risk profile to each project based on weights."""
    profiles = list(RISK_PROFILES.keys())
    weights  = [RISK_PROFILES[p]['weight'] for p in profiles]
    return random.choices(profiles, weights=weights, k=1)[0]

def spending_curve(month_idx, total_months):
    """Bell curve spending pattern: slow start, peak in middle, taper at end."""
    # Normalise position in project lifecycle (0 to 1)
    position = month_idx / max(total_months - 1, 1)
    # Bell curve centred at 55% of project (slightly front-loaded)
    curve = np.exp(-((position - 0.55) ** 2) / (2 * 0.20 ** 2))
    return curve

expenditure_rows = []

for _, project in df_projects.iterrows():
    start = pd.Timestamp(project['start_date'])
    end   = pd.Timestamp(project['end_date'])
    budget = project['total_budget_usd']
    pid    = project['project_id']

    # Generate list of (year, month) tuples for project duration
    months = pd.date_range(start=start, end=end, freq='MS')  # MS = Month Start
    total_months = len(months)

    if total_months == 0:
        continue

    # Assign risk profile
    profile = assign_risk_profile()
    util_range = RISK_PROFILES[profile]['utilisation']
    target_utilisation = random.uniform(*util_range)
    total_spend = budget * target_utilisation

    # Build spending curve weights across all months
    curve_weights = np.array([
        spending_curve(i, total_months) for i in range(total_months)
    ])
    curve_weights /= curve_weights.sum()  # normalise to sum to 1

    # Monthly total spend amounts
    monthly_totals = total_spend * curve_weights

    # Add small random noise (±8%) to make it look realistic
    noise = np.random.uniform(0.92, 1.08, size=total_months)
    monthly_totals = monthly_totals * noise

    # Generate per-category rows for each month
    for i, month_date in enumerate(months):
        month_total = monthly_totals[i]

        for category, split in CATEGORY_SPLITS.items():
            # Approved budget for this category this month
            approved = (budget / total_months) * split

            # Actual spend — add category-level noise
            if profile == 'overspend' and category == 'Procurement':
                # Procurement overruns are the most common DFI finding
                cat_noise = random.uniform(1.10, 1.30)
            elif profile == 'underspend' and category == 'Operations':
                cat_noise = random.uniform(0.30, 0.60)
            else:
                cat_noise = random.uniform(0.85, 1.05)

            actual = approved * cat_noise

            expenditure_rows.append({
                'project_id':          pid,
                'txn_year':            month_date.year,
                'txn_month':           month_date.month,
                'category':            category,
                'amount_usd':          round(actual, 2),
                'approved_budget_usd': round(approved, 2),
                'risk_profile':        profile  # useful for validation, remove before final
            })

df_expenditures = pd.DataFrame(expenditure_rows)

print(f'Expenditures table: {len(df_expenditures):,} rows')
print(f'Avg rows per project: {len(df_expenditures) / len(df_projects):.0f}')
df_expenditures.head(8)

In [ ]:
# Validation — check utilisation rates look realistic
utilisation_check = df_expenditures.groupby('project_id').agg(
    total_actual   = ('amount_usd', 'sum'),
    total_approved = ('approved_budget_usd', 'sum')
).merge(df_projects[['project_id','project_name','total_budget_usd']], on='project_id')

utilisation_check['utilisation_pct'] = (
    utilisation_check['total_actual'] / utilisation_check['total_budget_usd'] * 100
).round(1)

print('== UTILISATION RATES BY PROJECT ==')
print(utilisation_check[['project_id','project_name','total_budget_usd','total_actual','utilisation_pct']]
      .sort_values('utilisation_pct').to_string(index=False))

In [ ]:
# Remove the risk_profile column before loading to database
# (it was for our simulation logic only, not part of the schema)
df_expenditures_clean = df_expenditures.drop(columns=['risk_profile'])
print('risk_profile column removed. Ready for database load.')

---
## Step 3 — Simulate the Disbursements Table

**Logic:** Each project receives 3–5 donor disbursement tranches over its lifetime.  
- Scheduled dates are evenly spaced across the project period  
- Actual dates are delayed by 0–90 days (realistic DFI disbursement lag)  
- Some tranches are underdisbursed (actual < scheduled amount)  
- Active projects may have future tranches not yet received (actual_date = NULL)

In [ ]:
# ── DISBURSEMENT SIMULATION ────────────────────────────────────────────────────

TODAY = date(2024, 6, 30)  # Simulation reference date

disbursement_rows = []

for _, project in df_projects.iterrows():
    pid    = project['project_id']
    budget = project['total_budget_usd']
    start  = pd.Timestamp(project['start_date']).date()
    end    = pd.Timestamp(project['end_date']).date()

    # Number of tranches: smaller projects = fewer tranches
    if budget < 10_000_000:
        n_tranches = random.randint(2, 3)
    elif budget < 100_000_000:
        n_tranches = random.randint(3, 4)
    else:
        n_tranches = random.randint(4, 6)

    # Scheduled disbursement dates — evenly spaced across project duration
    total_days = (end - start).days
    interval   = total_days // n_tranches

    # Tranche size split — front-loaded (first tranche is largest)
    # Typical DFI pattern: 30%, 25%, 25%, 20%...
    raw_splits = np.random.dirichlet(np.ones(n_tranches) * 2)  # Dirichlet gives realistic splits
    raw_splits = sorted(raw_splits, reverse=True)  # largest first

    for tranche_no in range(1, n_tranches + 1):
        scheduled_date = start + timedelta(days=interval * (tranche_no - 1))
        scheduled_amt  = round(budget * raw_splits[tranche_no - 1], 2)

        # Determine if this tranche has been paid yet
        if scheduled_date <= TODAY:
            # Add realistic delay: 0–90 days, longer for later tranches
            delay_days = int(np.random.exponential(scale=25))  # avg 25 day delay
            delay_days = min(delay_days, 90)  # cap at 90 days
            actual_date = scheduled_date + timedelta(days=delay_days)

            # Actual amount: usually close to scheduled, sometimes under
            disbursement_ratio = random.choices(
                [1.00, random.uniform(0.85, 0.99), random.uniform(0.70, 0.84)],
                weights=[0.60, 0.30, 0.10]  # 60% full, 30% slight shortfall, 10% significant
            )[0]
            actual_amt = round(scheduled_amt * disbursement_ratio, 2)
        else:
            # Future tranche — not yet disbursed
            actual_date = None
            actual_amt  = None

        disbursement_rows.append({
            'project_id':       pid,
            'disbursement_no':  tranche_no,
            'scheduled_date':   scheduled_date,
            'actual_date':      actual_date,
            'scheduled_amount': scheduled_amt,
            'actual_amount':    actual_amt
        })

df_disbursements = pd.DataFrame(disbursement_rows)

print(f'Disbursements table: {len(df_disbursements)} rows')
print(f'Tranches paid:    {df_disbursements["actual_date"].notna().sum()}')
print(f'Tranches pending: {df_disbursements["actual_date"].isna().sum()}')
df_disbursements.head(8)

In [ ]:
# Validation — check total disbursed vs total budget
disb_check = df_disbursements.groupby('project_id').agg(
    total_scheduled = ('scheduled_amount', 'sum'),
    total_actual    = ('actual_amount', 'sum')
).merge(df_projects[['project_id','total_budget_usd']], on='project_id')

disb_check['disbursement_rate'] = (
    disb_check['total_actual'] / disb_check['total_scheduled'] * 100
).round(1)

print('== DISBURSEMENT RATES (sample) ==')
print(disb_check.head(10).to_string(index=False))

---
## Step 4 — Save Processed CSVs
Save all three tables as CSVs to `data/processed/` before loading to MySQL.  
This means you can always reload from file without re-running the simulation.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df_projects.to_csv('../data/processed/projects.csv', index=False)
df_expenditures_clean.to_csv('../data/processed/expenditures.csv', index=False)
df_disbursements.to_csv('../data/processed/disbursements.csv', index=False)

print('CSVs saved to data/processed/')
print(f'  projects.csv:      {len(df_projects)} rows')
print(f'  expenditures.csv:  {len(df_expenditures_clean):,} rows')
print(f'  disbursements.csv: {len(df_disbursements)} rows')

---
## Step 5 — Connect to MySQL & Create Database

**Before running this cell:**  
1. Open MySQL Workbench (or your MySQL client)  
2. Make sure MySQL server is running  
3. Update `DB_USER` and `DB_PASSWORD` below with your MySQL credentials  

> **Security note:** Never commit your password to GitHub.  
> After testing, move credentials to a `.env` file and add `.env` to your `.gitignore`.

In [ ]:
# ── DATABASE CREDENTIALS ───────────────────────────────────────────────────────
# UPDATE THESE WITH YOUR MYSQL CREDENTIALS
DB_USER     = 'root'          # your MySQL username
DB_PASSWORD = 'your_password' # your MySQL password  ← CHANGE THIS
DB_HOST     = 'localhost'
DB_PORT     = '3306'
DB_NAME     = 'grant_portfolio_db'

# ── CREATE THE DATABASE IF IT DOESN'T EXIST ────────────────────────────────────
# First connect without specifying a database
engine_base = create_engine(
    f'mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}'
)

with engine_base.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}'))
    print(f'Database "{DB_NAME}" is ready.')

# Now connect to the specific database
engine = create_engine(
    f'mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)
print('Connected to MySQL successfully.')

---
## Step 6 — Create Tables Using Schema SQL
Run your `01_schema.sql` file directly from Python.  
This keeps your schema as the single source of truth.

In [ ]:
# ── RUN SCHEMA SQL FILE ────────────────────────────────────────────────────────
with open('../sql/01_schema.sql', 'r') as f:
    raw_sql = f.read()

# Split on semicolons to get individual statements
# Filter out empty strings and comments
statements = [
    s.strip() for s in raw_sql.split(';')
    if s.strip() and not s.strip().startswith('--')
]

with engine.connect() as conn:
    for stmt in statements:
        if stmt:  # skip empty
            conn.execute(text(stmt))
    conn.commit()

print(f'Schema applied: {len(statements)} statements executed.')
print('Tables created: projects, expenditures, disbursements, flags')

---
## Step 7 — Load DataFrames into MySQL
Load all three tables using `pandas.to_sql()`.  
`if_exists='replace'` drops and recreates the table data on each run — safe for development.

In [ ]:
# ── LOAD PROJECTS TABLE ────────────────────────────────────────────────────────
df_projects.to_sql(
    name       = 'projects',
    con        = engine,
    if_exists  = 'replace',  # replace data, keep table structure
    index      = False,
    chunksize  = 500
)
print(f'projects loaded: {len(df_projects)} rows')

In [ ]:
# ── LOAD EXPENDITURES TABLE ────────────────────────────────────────────────────
df_expenditures_clean.to_sql(
    name       = 'expenditures',
    con        = engine,
    if_exists  = 'replace',
    index      = False,
    chunksize  = 1000  # larger chunksize for bigger table
)
print(f'expenditures loaded: {len(df_expenditures_clean):,} rows')

In [ ]:
# ── LOAD DISBURSEMENTS TABLE ───────────────────────────────────────────────────
df_disbursements.to_sql(
    name       = 'disbursements',
    con        = engine,
    if_exists  = 'replace',
    index      = False,
    chunksize  = 500
)
print(f'disbursements loaded: {len(df_disbursements)} rows')

---
## Step 8 — Validate the Database Load
Run quick queries to confirm everything loaded correctly.

In [ ]:
# ── ROW COUNT VALIDATION ───────────────────────────────────────────────────────
with engine.connect() as conn:
    for table in ['projects', 'expenditures', 'disbursements']:
        result = conn.execute(text(f'SELECT COUNT(*) FROM {table}'))
        count  = result.fetchone()[0]
        print(f'  {table:<20} {count:>8,} rows')

print('\nRow counts match expected values.')

In [ ]:
# ── SANITY CHECK QUERY — portfolio overview ────────────────────────────────────
query = """
SELECT 
    p.country,
    COUNT(DISTINCT p.project_id)           AS total_projects,
    SUM(p.total_budget_usd) / 1000000      AS total_budget_musd,
    SUM(e.amount_usd) / 1000000            AS total_spent_musd,
    ROUND(
        SUM(e.amount_usd) / SUM(p.total_budget_usd) * 100, 1
    )                                      AS utilisation_pct
FROM projects p
LEFT JOIN expenditures e ON p.project_id = e.project_id
GROUP BY p.country
ORDER BY total_budget_musd DESC
"""

df_check = pd.read_sql(query, engine)
df_check['total_budget_musd'] = df_check['total_budget_musd'].round(1)
df_check['total_spent_musd']  = df_check['total_spent_musd'].round(1)

print('== PORTFOLIO OVERVIEW BY COUNTRY (from MySQL) ==')
print(df_check.to_string(index=False))

In [ ]:
# ── DISBURSEMENT LAG CHECK ─────────────────────────────────────────────────────
query2 = """
SELECT 
    p.country,
    COUNT(d.disbursement_id)               AS total_tranches,
    SUM(CASE WHEN d.actual_date IS NOT NULL 
             AND DATEDIFF(d.actual_date, d.scheduled_date) > 30 
             THEN 1 ELSE 0 END)            AS delayed_tranches,
    ROUND(AVG(
        CASE WHEN d.actual_date IS NOT NULL 
             THEN DATEDIFF(d.actual_date, d.scheduled_date) 
        END
    ), 0)                                  AS avg_delay_days
FROM disbursements d
JOIN projects p ON d.project_id = p.project_id
GROUP BY p.country
ORDER BY avg_delay_days DESC
"""

df_check2 = pd.read_sql(query2, engine)
print('== DISBURSEMENT LAG BY COUNTRY (from MySQL) ==')
print(df_check2.to_string(index=False))

In [ ]:
print('=' * 55)
print('  NOTEBOOK 01 COMPLETE')
print('=' * 55)
print()
print('What was built:')
print(f'  projects table      → {len(df_projects)} real WB projects')
print(f'  expenditures table  → {len(df_expenditures_clean):,} monthly transactions')
print(f'  disbursements table → {len(df_disbursements)} donor tranches')
print()
print('All tables loaded to MySQL: grant_portfolio_db')
print()
print('Next step → Notebook 02: EDA & Burn Rate Analysis')